# Membuat Edges (Hubungan Guru dan Murid)

## Import Library

In [56]:
import pandas as pd
import ast
import re

## Load Data

In [57]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)
df.columns = ['Hadith', 'Book', 'Num_hadith', 'Matn', 'Sanad', 'Sanad_Length', 'Sanad_no_harakat']
df = df.iloc[1:].reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24128\1150346939.py:3: DtypeWarning: Columns (0: 2, 1: 5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)


### Menghitung data dan Memecah isi list pada kolom Sanad_no_harakat

In [58]:
df['Sanad_no_harakat'] = df['Sanad_no_harakat'].apply(ast.literal_eval)


In [59]:
df.shape

(487451, 7)

In [60]:
print(type(df['Sanad_no_harakat'].iloc[0]))

<class 'list'>


In [61]:
sanad_exploded = df.explode('Sanad_no_harakat')


In [62]:
sanad_exploded.shape

(2993324, 7)

In [63]:
df.Sanad_no_harakat.head(10)

0    [ابو عبيدة مسلم بن ابو كريمة التميمي, جابر بن ...
1                      [ابو عبيدة, جابر بن زيد, عائشة]
2                             [ابو عبيدة, جابر بن زيد]
3                  [ابو عبيدة, جابر بن زيد, ابو هريرة]
4            [ابو عبيدة, جابر بن زيد, ابو سعيد الخدري]
5                   [ابو عبيدة, جابر بن زيد, ابن عباس]
6                [ابو عبيدة, جابر بن زيد, انس بن مالك]
7            [ابو عبيدة, جابر بن زيد, ابو سعيد الخدري]
8                  [ابو عبيدة, جابر بن زيد, ابو هريرة]
9              [ابو عبيدة, جابر بن زيد, عمر بن الخطاب]
Name: Sanad_no_harakat, dtype: object

### Menghitung Perawi (Narrator) Paling Banyak Muncul

In [64]:
# Hitung frekuensi setiap perawi
from collections import Counter

# Flatten semua nama perawi dari list Sanad_no_harakat
all_narrators = []
for sanad_list in df['Sanad_no_harakat']:
    all_narrators.extend(sanad_list)

# Hitung frekuensi
narrator_counts = Counter(all_narrators)

# Buat DataFrame dari hasil perhitungan
narrator_df = pd.DataFrame(narrator_counts.most_common(), columns=['Perawi', 'Jumlah'])
narrator_df

,Perawi,Jumlah
0,ابو هريرة,51623
1,ابن عباس,47925
2,عائشة,33019
3,ابن عمر,28671
4,شعبة,26324
...,...,...
191599,الاشعث بن طلق,1
191600,وحفص بن موسي,1
191601,الحجاج <IDF> يعني<IDF> الصواف,1
191602,وابي مسعود البدري,1


### Buat Hubungan Guru-Murid dari Sanad


In [65]:
from IPython.display import display

# Fungsi membentuk hubungan guru-murid
def build_teacher_student_edges(chain):
    if not isinstance(chain, list) or len(chain) < 2:
        return []
    
    # Format: (Guru, Murid)
    return [(chain[i + 1], chain[i]) for i in range(len(chain) - 1)]

# Pastikan semua data berupa list
def ensure_list(cell):
    if isinstance(cell, list):
        return cell
    
    if isinstance(cell, str) and cell.strip().startswith('['):
        try:
            return ast.literal_eval(cell)
        except Exception:
            return [cell]
    
    if pd.isna(cell):
        return []
    
    return [cell]

# Ubah kolom menjadi list
sanad_lists = df['Sanad_no_harakat'].apply(ensure_list)

# Membentuk semua hubungan guru-murid
edges = []
for sanad in sanad_lists:
    edges.extend(build_teacher_student_edges(sanad))

# Membuat DataFrame hubungan
edges_df = pd.DataFrame(edges, columns=['Guru', 'Murid'])

# Hitung weight tanpa mengubah urutan awal
edges_df['Weight'] = (
    edges_df
    .groupby(['Guru', 'Murid'])['Guru']
    .transform('count')
)

# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight']])

,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيدة,522
3,عائشة,جابر بن زيد,68
4,ابو هريرة,جابر بن زيد,76
...,...,...,...
806654,وابي مسعود البدري,حذيفة بن اليمان,1
806655,اهل الكوفة الي سعيد بن العاص,وابي مسعود البدري,1
806656,ابو مسعود,اهل الكوفة الي سعيد بن العاص,1
806657,عبد الله,علي بن علقمة,1


In [66]:
type(df['Sanad_no_harakat'].iloc[0])

list

In [67]:
type(edges_df['Guru'].iloc[0])

str

### Menghapus Lafal Periwayatan

In [68]:
# Daftar lafal periwayatan yang dihapus
lafal_periwayatan = [
    # Bentuk dasar
    'حدثنا', 'حدثني', 'حدث', 'حدثت',
    'اخبرنا', 'اخبرني', 'اخبر',
    'أخبرنا', 'أخبرني', 'أخبر',

    # Anba'
    'أنبأنا', 'انبانا',
    'أنبأني', 'انبأني',
    'انباني',

    # Qala dan turunannya
    'قال', 'قلت', 'يقول',
    'فقال', 'وقال', 'قالت',

    # Sami‘a
    'سمعت', 'يسمع',

    # Riwayat
    'روى', 'يروي',

    # Kata penghubung sanad
    'عن', 'ذكر',

    # Singkatan sanad
    'ثنا', 'نا',

    # Tambahan umum sanad
    'قرأت', 'قرأنا',
    'كتب', 'كتب إلي',
    'أملاء', 'إملاء',
    'سألت', 'سألنا',
    'حدثكم', 'حدثهم',
    'اخبركم', 'أخبركم',
    'أن', 'إن'
]

noise_patterns = [
    r'اهل الكوفة',
    r'اهل البصرة',
    r'اهل المدينة',
    r'اهل مكة',
    r'\bالي\b'
]

# Gabungkan lafal menjadi regex
lafal_pattern = r'\b(?:' + '|'.join(lafal_periwayatan) + r')\b'
noise_pattern = '|'.join(noise_patterns)

# Karakter/simbol yang dihapus
pattern = r'[,،#()="{}\-\?\.:\|]+|<IDF>|</IDF>'


# Fungsi cleaning
def clean_text(text):

    if not isinstance(text, str):
        return text

    # Hapus lafal periwayatan
    text = re.sub(lafal_pattern, ' ', text)

    # Hapus noise tambahan
    text = re.sub(noise_pattern, ' ', text)

    # Hapus harakat Arab
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)

    # Hapus tanda baca Arab dan simbol
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    
    # Hapus waw di awal kata
    text = re.sub(r'^و(?=[\u0600-\u06FF])', '', text)


    # Hapus tatweel
    text = re.sub(r'ـ+', '', text)

    # Hapus karakter/simbol khusus
    text = re.sub(pattern, '', text)

    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus angka
    text = re.sub(r'\d+', '', text)

     # Hapus underscore
    text = text.replace('_', ' ')

    return text

# Terapkan ke kolom Guru dan Murid
for col in ['Guru', 'Murid']:
    edges_df[col] = edges_df[col].apply(clean_text)


# Tampilkan hasil
display(edges_df[['Guru', 'Murid', 'Weight']])

,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيدة,522
3,عائشة,جابر بن زيد,68
4,ابو هريرة,جابر بن زيد,76
...,...,...,...
806654,ابي مسعود البدري,حذيفة بن اليمان,1
806655,سعيد بن العاص,ابي مسعود البدري,1
806656,ابو مسعود,سعيد بن العاص,1
806657,عبد الله,علي بن علقمة,1


In [69]:
edges_df = edges_df.dropna(subset=['Guru', 'Murid'])

In [70]:
remove_text = "أن أم الفضل بنت الحارث بعثته إلى معاوية بالشام"

edges_df = edges_df[
    ~edges_df['Guru'].str.contains(remove_text, na=False) &
    ~edges_df['Murid'].str.contains(remove_text, na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(806659, 3)

In [71]:
edges_df = edges_df[
    ~edges_df['Guru'].str.contains(r'\d', na=False) &
    ~edges_df['Murid'].str.contains(r'\d', na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(806659, 3)

In [72]:
edges_df = edges_df[
    (edges_df['Guru'] != '') &
    (edges_df['Murid'] != '')
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(806515, 3)

In [73]:
edges_df = edges_df[
    edges_df['Guru'] != edges_df['Murid']
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(802063, 3)

In [74]:
# Daftar kata/lafaz periwayatan yang ingin dihapus barisnya
hapus_kata = [
    "ثنا", "حدثنا", "قال حدثنا", "وقال", "عن",
    "أخبرنا", "أنبأنا", "حدثني", "حدثه", "سمعت",
    "يقول", "ذكر", "رواه", "حدث", "حكى"
]

# Buat pola regex dari list hapus_kata (pakai word boundary biar lebih akurat)
pola = r'\b(?:' + '|'.join(hapus_kata) + r')\b'

# Filter baris yang TIDAK mengandung kata-kata itu di Murid maupun Guru
edges_df = edges_df[
    ~edges_df['Murid'].str.contains(pola, na=False) &
    ~edges_df['Guru'].str.contains(pola, na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(801793, 3)

In [75]:
# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Tambahkan inverse weight
edges_df["Inverse_Weight"] = (
    edges_df["Weight"]
    .apply(lambda x: 1/x if x > 0 else 0)
)

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيدة,522,0.001916
3,عائشة,جابر بن زيد,68,0.014706
4,ابو هريرة,جابر بن زيد,76,0.013158
...,...,...,...,...
776990,ابي مسعود البدري,حذيفة بن اليمان,1,1.000000
776991,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776992,ابو مسعود,سعيد بن العاص,1,1.000000
776993,عبد الله,علي بن علقمة,1,1.000000


In [76]:
def clean_text(s):
    if pd.isna(s):
        return s
    
    # hapus karakter non-printable TAPI jangan hapus huruf Arab
    s = re.sub(r"[^\x20-\x7E\u0600-\u06FF\s]", "", str(s))
    # rapikan spasi
    return re.sub(r"\s+", " ", s).strip()

edges_df['Guru'] = edges_df['Guru'].apply(clean_text)
edges_df['Murid'] = edges_df['Murid'].apply(clean_text)

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيدة مسلم بن ابو كريمة التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيدة,522,0.001916
3,عائشة,جابر بن زيد,68,0.014706
4,ابو هريرة,جابر بن زيد,76,0.013158
...,...,...,...,...
776990,ابي مسعود البدري,حذيفة بن اليمان,1,1.000000
776991,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776992,ابو مسعود,سعيد بن العاص,1,1.000000
776993,عبد الله,علي بن علقمة,1,1.000000


In [77]:
print(edges_df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [80]:
edges_df.to_csv("../data/processed/edges.csv", index=False, encoding='utf-8-sig')